In [13]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import pickle
import pandas as pd
from utils import tld_bucket


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kaggleprollc/phishing-url-websites-dataset-phiusiil")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'phishing-url-websites-dataset-phiusiil' dataset.
Path to dataset files: /kaggle/input/phishing-url-websites-dataset-phiusiil


In [5]:
df=pd.read_csv("/kaggle/input/phishing-url-websites-dataset-phiusiil/PhiUSIIL_Phishing_URL_Dataset.csv")

In [6]:

pd.set_option('display.max_columns', None)

In [9]:
keep_cols = [
    'URLLength', 'DomainLength', 'IsDomainIP', 'TLD', 'TLDLength',
    'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar',
    'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL',
    'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL',
    'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL', 'IsHTTPS', 'label'
]

df = df[keep_cols]

In [43]:
df.to_csv('cleaned_data.csv', index=False)

In [10]:
df.head()

,URLLength,DomainLength,IsDomainIP,TLD,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,label
0,31,24,0,com,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,1
1,23,16,0,de,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,1
2,29,22,0,uk,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,1
3,26,19,0,com,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,1
4,33,26,0,org,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,1


In [12]:
df['TLD_risk'] = df['TLD'].apply(tld_bucket)
df.drop(columns=['TLD'], inplace=True)

In [14]:
df.head()

,URLLength,DomainLength,IsDomainIP,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,label,TLD_risk
0,31,24,0,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,1,0
1,23,16,0,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,1,0
2,29,22,0,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,1,0
3,26,19,0,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,1,0
4,33,26,0,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,1,0


In [15]:
df.duplicated().sum()

np.int64(203022)

In [16]:
df.drop_duplicates(inplace=True)

In [17]:
df.shape

(32773, 20)

In [18]:
X=df.drop(columns=['label'])
y=df['label']


In [19]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [20]:
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(26218, 19) (6555, 19) (26218,) (6555,)


In [21]:
preprocess=ColumnTransformer([
    ("num",Pipeline([
        ('scaler',StandardScaler())
    ]),x_train.columns)
])

In [23]:
pipe=make_pipeline(preprocess,SVC(kernel='rbf',class_weight='balanced',probability=True))

In [24]:
pipe.fit(x_train,y_train)


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['URLLength', 'DomainLength', 'IsDomainIP', 'TLDLength', 'NoOfSubDomain',
       'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
       'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL',
       'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL',
       'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL',
       'SpacialCharRatioInURL', 'IsHTTPS', 'TLD_risk'],
      dtype='object'))])),
                ('svc', SVC(class_weight='balanced', probability=True))])

In [25]:
prediction=pipe.predict(x_test)
prediction_prob=pipe.predict_proba(x_test)[:, 1]
prediction= (prediction_prob> 0.3).astype(int)
prediction_report=classification_report(y_test,prediction,output_dict=True)
prediction_df=pd.DataFrame(prediction_report).transpose()

In [38]:
print("Accuracy Score:",accuracy_score(y_test,prediction))

Accuracy Score: 0.9920671243325706


In [40]:
y_train_pred = pipe.predict(x_train)
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test,prediction))

Train Accuracy: 0.9843237470440156
Test Accuracy: 0.9920671243325706


In [39]:
scores = cross_val_score(pipe, X, y, cv=10)

print(scores)
print(scores.mean())

[0.97925564 0.9838316  0.9838316  0.98626793 0.98260604 0.98352151
 0.98779371 0.98748856 0.98443699 0.98352151]
0.9842555105629247


In [26]:
prediction_df

,precision,recall,f1-score,support
0,0.999842,0.991999,0.995905,6374.000000
1,0.779221,0.994475,0.873786,181.000000
accuracy,0.992067,0.992067,0.992067,0.992067
macro avg,0.889531,0.993237,0.934846,6555.000000
weighted avg,0.993750,0.992067,0.992533,6555.000000


In [35]:
conf=pd.DataFrame(confusion_matrix(y_test,prediction), columns=['Predicted Legit','Predicted Phishing'],index=['True Legit','True Phishing'])


In [36]:
conf

,Predicted Legit,Predicted Phishing
True Legit,6323,51
True Phishing,1,180


In [41]:
# Save the trained SVM model to a file
filename = 'Phish_guard_model.pkl'
pickle.dump(pipe, open(filename, 'wb'))

print(f"SVM model successfully saved to {filename}")

SVM model successfully saved to Phish_guard_model.pkl
